# Werkcollege-opdrachten Week 1.2

## Voorbereiding

Importeer in het codeblok hieronder de packages die worden gebruikt om data in te lezen. Geef er ook de gebruikelijke aliassen aan.<br>
N.B.: de 2 reeds geschreven coderegels zorgen ervoor dat eventuele warnings, die code-outputs lelijker maken, uitgezet worden.

In [274]:

import warnings
warnings.simplefilter('ignore')
import pandas as pd
import sqlite3

Zet de volgende bestanden in een makkelijk terug te vinden map:
- go_sales_train.sqlite
- go_crm_train.sqlite
- go_staff_train.sqlite

Bestudeer de bovenste 3 bestanden in DB Browser (SQLite), <a href="https://sqlitebrowser.org/dl/">hier</a> te downloaden. Wat valt je op qua datatypen?<br>

## Databasetabel inlezen

Creëer een databaseconnectie met het bestand go_sales_train.sqlite.

In [275]:
conn = sqlite3.connect("..\Data\Week1\go_sales_train.sqlite")
print(conn)

<b>Let goed op:</b><br>
Als je per ongeluk een verkeerde bestandsnaam ingeeft, maakt Python zélf een leeg databasebestand aan! Er ontstaat dan dus een nieuwe .sqlite, en dat is nadrukkelijk <u>niet de bedoeling.</u>

Gebruik de onderstaande sql_query om te achterhalen welke databasetabellen in go_sales_train zitten.

In [276]:
sql_query = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(sql_query)
#Vul dit codeblok verder in

             name
0         country
1   order_details
2    order_header
3    order_method
4         product
5    product_line
6    product_type
7   retailer_site
8   return_reason
9   returned_item
10   sales_branch
11    sales_staff


Krijg je lege output? Dan heb je per ongeluk een leeg  databasebestand (.sqlite) aangemaakt.<br>
Lees de informatie onder het kopje <u>Let goed op:</u> hierboven nog eens goed door.

Gebruik de gecreëerde databaseconnectie om de resultaten van de volgende query in een DataFrame te zetten:<br>
*SELECT * FROM sales_staff* 

In [277]:
sql_query = "SELECT * FROM sales_staff"
data_frame = pd.read_sql(sql_query, conn)

print(data_frame)

     SALES_STAFF_CODE FIRST_NAME  LAST_NAME                   POSITION_EN  \
0                   4      Denis       Pagé                Branch Manager   
1                   5  Élizabeth     Michel  Level 3 Sales Representative   
2                   6      Émile   Clermont  Level 1 Sales Representative   
3                   7    Étienne     Jauvin  Level 2 Sales Representative   
4                  12    Elsbeth  Wiesinger  Level 2 Sales Representative   
..                ...        ...        ...                           ...   
97                120      Giele   Laermans  Level 1 Sales Representative   
98                121   François    De Crée  Level 1 Sales Representative   
99                122     Yvette    Lattrez  Level 3 Sales Representative   
100               123      Willi  Seefelder  Level 2 Sales Representative   
101               124     Sabine     Grüner  Level 3 Sales Representative   

            WORK_PHONE  EXTENSION                FAX                   EMAI

## Datumkolommen

Zoals je misschien al hebt gezien in DB Browser, zijn datums als TEXT opgeslagen, en niet als DATE, DATETIME o.i.d. Hier moeten we dus nog even "typische datumkolommen" van maken. Dat doen we met de volgende code:

In [278]:
data_frame['DATE_HIRED'] = pd.to_datetime(data_frame['DATE_HIRED'])
data_frame.dtypes

SALES_STAFF_CODE              int64
FIRST_NAME                   object
LAST_NAME                    object
POSITION_EN                  object
WORK_PHONE                   object
EXTENSION                   float64
FAX                          object
EMAIL                        object
DATE_HIRED           datetime64[ns]
SALES_BRANCH_CODE             int64
dtype: object

Als we hier het jaar uit willen halen, schrijven we:

In [279]:
print(data_frame['DATE_HIRED'].dt.year)

0      1996
1      1995
2      1998
3      1997
4      1997
       ... 
97     1999
98     1999
99     1999
100    1998
101    1998
Name: DATE_HIRED, Length: 102, dtype: int32


Deze zelfde syntax is te gebruiken voor het extraheren van kwartalen, maanden, weken en dagen. Probeer het maar eens!

## DataFrames uitsplitsen en opbouwen met Series

De volgende 5 kolommen hebben betrekking op de contactdetails van elke medewerker in dit DataFrame:
- SALES_STAFF_CODE
- WORK_PHONE
- EXTENSION
- FAX
- EMAIL

Maak van elk van deze 5 kolommen een serie.

In [280]:
sales_staff_codeSerie = data_frame['SALES_STAFF_CODE']
work_phoneSerie = data_frame['WORK_PHONE']
extensionSerie = data_frame['EXTENSION']
faxSerie = data_frame['FAX']
emailSerie = data_frame['EMAIL']

Zet allevijf gecreëerde series als kolommen naast elkaar in een DataFrame (*contact_details*). Gebruik pd.concat(...)<br>
Hulpvraag: welke waarde geef je aan de axis-parameter?

In [281]:
modified_Dataframe = pd.concat([sales_staff_codeSerie, work_phoneSerie, extensionSerie, faxSerie, emailSerie], axis=1)
print(modified_Dataframe)

     SALES_STAFF_CODE         WORK_PHONE  EXTENSION                FAX  \
0                   4  +33 1 68 94 52 20      325.0  +33 1 68 94 56 60   
1                   5  +33 1 68 94 52 20      336.0  +33 1 68 94 56 60   
2                   6  +33 1 68 94 52 20      378.0  +33 1 68 94 56 60   
3                   7  +33 1 68 94 52 20      398.0  +33 1 68 94 56 60   
4                  12  +(49) 40 663 1990     1818.0  +(49) 40 663 4571   
..                ...                ...        ...                ...   
97                120    +32 16 20.73.21     1340.0    +32 16 20.73.32   
98                121    +32 16 20.73.21     1642.0    +32 16 20.73.32   
99                122    +32 16 20.73.21     1633.0    +32 16 20.73.32   
100               123  +(43) 13 79 56 32      325.0  +(43) 13 79 56 33   
101               124  +(43) 13 79 56 32      348.0  +(43) 13 79 56 33   

                      EMAIL  
0         DPage@grtd123.com  
1       EMichel@grtd123.com  
2     EClermont@grtd1

## Series en DataFrames maken vanuit lists en dictionaries

Met .head(*getal*) kan je de bovenste *getal* rijen van een tabel selecteren.<br>
Selecteer op deze manier de bovenste 5 rijen van *contact_details*.<br>
Sla dit resultaat op in een nieuw DataFrame.

In [282]:
modified_Dataframe2 = modified_Dataframe
print(modified_Dataframe2.head(5))

   SALES_STAFF_CODE         WORK_PHONE  EXTENSION                FAX  \
0                 4  +33 1 68 94 52 20      325.0  +33 1 68 94 56 60   
1                 5  +33 1 68 94 52 20      336.0  +33 1 68 94 56 60   
2                 6  +33 1 68 94 52 20      378.0  +33 1 68 94 56 60   
3                 7  +33 1 68 94 52 20      398.0  +33 1 68 94 56 60   
4                12  +(49) 40 663 1990     1818.0  +(49) 40 663 4571   

                    EMAIL  
0       DPage@grtd123.com  
1     EMichel@grtd123.com  
2   EClermont@grtd123.com  
3     EJauvin@grtd123.com  
4  EWiesinger@grtd123.com  


Aan deze 10 rijen met contactdetails willen we 3 kolommen toevoegen: 'FIRST_LANGUAGE', 'SECOND_LANGUAGE' & 'THIRD_LANGUAGE'.<br>
Iedereens 'First Language' is Engels, afgekort 'EN'. Maak een lijst waarin 5x 'EN' staat.<br>
Converteer deze lijst vervolgens naar een serie en voeg deze horizontaal samen met het resultaat van de vorige opdracht.<br>
Vergeet niet een passende naam te geven aan de nieuw ontstane kolom.

In [283]:
language_list = ['EN', 'EN', 'EN', 'EN', 'EN']

first_languageSerie = pd.Series(language_list, name='FIRST_LANGUAGE')

modified_Dataframe2 = pd.concat([modified_Dataframe2, first_languageSerie], axis=1)
print(modified_Dataframe2.head(5))

   SALES_STAFF_CODE         WORK_PHONE  EXTENSION                FAX  \
0                 4  +33 1 68 94 52 20      325.0  +33 1 68 94 56 60   
1                 5  +33 1 68 94 52 20      336.0  +33 1 68 94 56 60   
2                 6  +33 1 68 94 52 20      378.0  +33 1 68 94 56 60   
3                 7  +33 1 68 94 52 20      398.0  +33 1 68 94 56 60   
4                12  +(49) 40 663 1990     1818.0  +(49) 40 663 4571   

                    EMAIL FIRST_LANGUAGE  
0       DPage@grtd123.com             EN  
1     EMichel@grtd123.com             EN  
2   EClermont@grtd123.com             EN  
3     EJauvin@grtd123.com             EN  
4  EWiesinger@grtd123.com             EN  


Maak nu de tweede kolom ('SECOND_LANGUAGE'). Maak daarvoor een dictionary, met daarin...
- Als keys: alle indexen uit het resultaat van het vorige codeblok.
- Als values: bij de eerste 3 elementen 'FR' (Frankrijk), bij de laatste 2 'DE' (Duitsland).

Maak vervolgens ook hier weer een serie van en voeg ook deze weer horizontaal samen met het rsultaat van de vorige opdracht.<br>
Vergeet niet een passende naam te geven aan de nieuw ontstane kolom.

In [284]:
language_dic = {0 : 'FR', 1 : 'FR', 2 : 'FR', 3 : 'DE', 4 : 'DE'}

second_languageSerie = pd.Series(language_dic, name='SECOND_LANGUAGE')

modified_Dataframe2 = pd.concat([modified_Dataframe2, second_languageSerie], axis=1)
print(modified_Dataframe2.head(5))

   SALES_STAFF_CODE         WORK_PHONE  EXTENSION                FAX  \
0                 4  +33 1 68 94 52 20      325.0  +33 1 68 94 56 60   
1                 5  +33 1 68 94 52 20      336.0  +33 1 68 94 56 60   
2                 6  +33 1 68 94 52 20      378.0  +33 1 68 94 56 60   
3                 7  +33 1 68 94 52 20      398.0  +33 1 68 94 56 60   
4                12  +(49) 40 663 1990     1818.0  +(49) 40 663 4571   

                    EMAIL FIRST_LANGUAGE SECOND_LANGUAGE  
0       DPage@grtd123.com             EN              FR  
1     EMichel@grtd123.com             EN              FR  
2   EClermont@grtd123.com             EN              FR  
3     EJauvin@grtd123.com             EN              DE  
4  EWiesinger@grtd123.com             EN              DE  


Maak ten slotte de derde kolom ('THIRD_LANGUAGE') door een dictionary te maken met daarin...
- Als key: de naam van de nieuwe kolom. Zie je het verschil qua keys met de vorige opdracht?
- Als waarde: een lijst met daarin 'NL', 'NL', 'JPN', 'JPN', 'KOR'.

Converteer deze dictionary nu naar een DataFrame en voeg deze horizontaal samen met het resultaat van de vorige opdracht.<br>
Waarom hoef je hierna de nieuw ontstane kolom geen passende naam meer te geven?

In [285]:
language_dic = {'THIRD_LANGUAGE': ['NL', 'NL', 'JPN', 'JPN', 'KOR']}
third_language_df = pd.DataFrame(language_dic)
modified_Dataframe2 = pd.concat([modified_Dataframe2, third_language_df], axis=1)

print(modified_Dataframe2.head(5))

   SALES_STAFF_CODE         WORK_PHONE  EXTENSION                FAX  \
0                 4  +33 1 68 94 52 20      325.0  +33 1 68 94 56 60   
1                 5  +33 1 68 94 52 20      336.0  +33 1 68 94 56 60   
2                 6  +33 1 68 94 52 20      378.0  +33 1 68 94 56 60   
3                 7  +33 1 68 94 52 20      398.0  +33 1 68 94 56 60   
4                12  +(49) 40 663 1990     1818.0  +(49) 40 663 4571   

                    EMAIL FIRST_LANGUAGE SECOND_LANGUAGE THIRD_LANGUAGE  
0       DPage@grtd123.com             EN              FR             NL  
1     EMichel@grtd123.com             EN              FR             NL  
2   EClermont@grtd123.com             EN              FR            JPN  
3     EJauvin@grtd123.com             EN              DE            JPN  
4  EWiesinger@grtd123.com             EN              DE            KOR  


## Data toevoegen

### Rijen

Gebruik de originele databasetabel *sales_staff*.<br>
Voeg een extra rij toe met eigen invulling. Zorg ervoor dat de index netjes doorloopt.<br>
Hulpvraag: welke waarde geef je nu aan axis?

In [286]:
sql_query = "SELECT * FROM sales_staff"
data_frame = pd.read_sql(sql_query, conn)

new_row = {
    'SALES_STAFF_CODE': 6969,
    'FIRST_NAME': 'Mark',
    'LAST_NAME': 'Wilbrink',
    'POSITION_EN': 'Sales Manager',
    'WORK_PHONE': '+1234567890',
    'EXTENSION': 123,
    'FAX': '+1234567890',
    'EMAIL': 'Mark@gmail.com',
    'DATE_HIRED': '1970-10-11',
    'SALES_BRANCH_CODE': 6
}

new_row_df = pd.DataFrame([new_row])

updated_data_frame = pd.concat([data_frame, new_row_df], ignore_index=True)

print(updated_data_frame.tail(5))

     SALES_STAFF_CODE FIRST_NAME  LAST_NAME                   POSITION_EN  \
98                121   François    De Crée  Level 1 Sales Representative   
99                122     Yvette    Lattrez  Level 3 Sales Representative   
100               123      Willi  Seefelder  Level 2 Sales Representative   
101               124     Sabine     Grüner  Level 3 Sales Representative   
102              6969       Mark   Wilbrink                 Sales Manager   

            WORK_PHONE  EXTENSION                FAX                   EMAIL  \
98     +32 16 20.73.21     1642.0    +32 16 20.73.32     FDecree@grtd123.com   
99     +32 16 20.73.21     1633.0    +32 16 20.73.32    YLattrez@grtd123.com   
100  +(43) 13 79 56 32      325.0  +(43) 13 79 56 33  WSeefelder@grtd123.com   
101  +(43) 13 79 56 32      348.0  +(43) 13 79 56 33     SGruner@grtd123.com   
102        +1234567890      123.0        +1234567890          Mark@gmail.com   

     DATE_HIRED  SALES_BRANCH_CODE  
98   1999-12-01    

### Kolommen

Voeg een kolom *FULL_NAME* toe die de waarden van *FIRST_NAME* en *LAST_NAME* achter elkaar zet, gescheiden door een spatie.

In [287]:
data_frame['FULL_NAME'] = data_frame['FIRST_NAME'] + ' ' + data_frame['LAST_NAME']
print(data_frame.head(5))

   SALES_STAFF_CODE FIRST_NAME  LAST_NAME                   POSITION_EN  \
0                 4      Denis       Pagé                Branch Manager   
1                 5  Élizabeth     Michel  Level 3 Sales Representative   
2                 6      Émile   Clermont  Level 1 Sales Representative   
3                 7    Étienne     Jauvin  Level 2 Sales Representative   
4                12    Elsbeth  Wiesinger  Level 2 Sales Representative   

          WORK_PHONE  EXTENSION                FAX                   EMAIL  \
0  +33 1 68 94 52 20      325.0  +33 1 68 94 56 60       DPage@grtd123.com   
1  +33 1 68 94 52 20      336.0  +33 1 68 94 56 60     EMichel@grtd123.com   
2  +33 1 68 94 52 20      378.0  +33 1 68 94 56 60   EClermont@grtd123.com   
3  +33 1 68 94 52 20      398.0  +33 1 68 94 56 60     EJauvin@grtd123.com   
4  +(49) 40 663 1990     1818.0  +(49) 40 663 4571  EWiesinger@grtd123.com   

   DATE_HIRED  SALES_BRANCH_CODE          FULL_NAME  
0  1996-11-03             

## Data wijzigen

### Datatypen

Door het attribuut .dtypes van een DataFrame op te vragen krijg je een serie die per kolom het datatype weergeeft. Doe dit bij de originele databasetabel *sales_staff*

In [288]:
sql_query = "SELECT * FROM sales_staff"
data_frame = pd.read_sql(sql_query, conn)

print(data_frame.dtypes)

SALES_STAFF_CODE       int64
FIRST_NAME            object
LAST_NAME             object
POSITION_EN           object
WORK_PHONE            object
EXTENSION            float64
FAX                   object
EMAIL                 object
DATE_HIRED            object
SALES_BRANCH_CODE      int64
dtype: object


Hier valt op dat elke kolom het datatype 'object' heeft: Python leest dus alles als string. Wiskundige operaties zijn hierdoor niet mogelijk.<br>
We kunnen proberen om kolommen met getallen, bijvoorbeeld de 'extension', te converteren naar een int. Zie onderstaande code.

In [289]:
data_frame['EXTENSION'] = data_frame['EXTENSION'].astype(int)
data_frame.dtypes

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

Dit lukt echter niet, omdat er in de kolom 'EXTENSION' lege waarden zitten die niet geconverteerd kunnen worden naar een int.<br>
Wél kan je deze naar een float converteren, zie onderstaande code:

In [ ]:
data_frame['EXTENSION'] = data_frame['EXTENSION'].astype(float)
data_frame.dtypes

SALES_STAFF_CODE       int64
FIRST_NAME            object
LAST_NAME             object
POSITION_EN           object
WORK_PHONE            object
EXTENSION            float64
FAX                   object
EMAIL                 object
DATE_HIRED            object
SALES_BRANCH_CODE      int64
dtype: object

Als we in de rij van 'EXTENSION' kijken zien we dat de conversie van het datatype nu is gelukt.<br>
Dit is <b>randvoorwaardelijk</b> voor het uitvoeren van wiskundige operaties.<br>

### Rijen

Zorg er nu voor dat bij alle extensions 1 wordt opgeteld.

In [ ]:
data_frame['EXTENSION'] += 1
print(data_frame)

     SALES_STAFF_CODE FIRST_NAME  LAST_NAME                   POSITION_EN  \
0                   4      Denis       Pagé                Branch Manager   
1                   5  Élizabeth     Michel  Level 3 Sales Representative   
2                   6      Émile   Clermont  Level 1 Sales Representative   
3                   7    Étienne     Jauvin  Level 2 Sales Representative   
4                  12    Elsbeth  Wiesinger  Level 2 Sales Representative   
..                ...        ...        ...                           ...   
97                120      Giele   Laermans  Level 1 Sales Representative   
98                121   François    De Crée  Level 1 Sales Representative   
99                122     Yvette    Lattrez  Level 3 Sales Representative   
100               123      Willi  Seefelder  Level 2 Sales Representative   
101               124     Sabine     Grüner  Level 3 Sales Representative   

            WORK_PHONE  EXTENSION                FAX                   EMAI

Elke 'Branch Manager' wordt nu 'General Manager'. Schrijf code zodat deze wijziging doorgevoerd wordt in het DataFrame.

In [ ]:
data_frame.loc[data_frame['POSITION_EN'] == 'Branch Manager', 'POSITION_EN'] = 'General Manager'
print(data_frame)

     SALES_STAFF_CODE FIRST_NAME  LAST_NAME                   POSITION_EN  \
0                   4      Denis       Pagé               General Manager   
1                   5  Élizabeth     Michel  Level 3 Sales Representative   
2                   6      Émile   Clermont  Level 1 Sales Representative   
3                   7    Étienne     Jauvin  Level 2 Sales Representative   
4                  12    Elsbeth  Wiesinger  Level 2 Sales Representative   
..                ...        ...        ...                           ...   
97                120      Giele   Laermans  Level 1 Sales Representative   
98                121   François    De Crée  Level 1 Sales Representative   
99                122     Yvette    Lattrez  Level 3 Sales Representative   
100               123      Willi  Seefelder  Level 2 Sales Representative   
101               124     Sabine     Grüner  Level 3 Sales Representative   

            WORK_PHONE  EXTENSION                FAX                   EMAI

### Kolommen

Verander de kolomnaam 'POSITION_EN' naar 'POSITION'.

In [ ]:
data_frame.rename(columns={'POSITION_EN': 'POSITION'}, inplace=True)
print(data_frame)


     SALES_STAFF_CODE FIRST_NAME  LAST_NAME                      POSITION  \
0                   4      Denis       Pagé               General Manager   
1                   5  Élizabeth     Michel  Level 3 Sales Representative   
2                   6      Émile   Clermont  Level 1 Sales Representative   
3                   7    Étienne     Jauvin  Level 2 Sales Representative   
4                  12    Elsbeth  Wiesinger  Level 2 Sales Representative   
..                ...        ...        ...                           ...   
97                120      Giele   Laermans  Level 1 Sales Representative   
98                121   François    De Crée  Level 1 Sales Representative   
99                122     Yvette    Lattrez  Level 3 Sales Representative   
100               123      Willi  Seefelder  Level 2 Sales Representative   
101               124     Sabine     Grüner  Level 3 Sales Representative   

            WORK_PHONE  EXTENSION                FAX                   EMAI

## Data verwijderen

### Rijen

De medewerkers op indexen 99, 100 en 101 hebben helaas ontslag genomen.<br>
Verwijder de bijbehorende rijen uit het DataFrame. Gebruik slechts één keer de .drop()-methode.

In [ ]:
data_frame.drop([99, 100, 101], inplace=True)
print(data_frame)

    SALES_STAFF_CODE FIRST_NAME  LAST_NAME                      POSITION  \
0                  4      Denis       Pagé               General Manager   
1                  5  Élizabeth     Michel  Level 3 Sales Representative   
2                  6      Émile   Clermont  Level 1 Sales Representative   
3                  7    Étienne     Jauvin  Level 2 Sales Representative   
4                 12    Elsbeth  Wiesinger  Level 2 Sales Representative   
..               ...        ...        ...                           ...   
94               117      Frank      Jever  Level 3 Sales Representative   
95               118     Gianni  Vertemati  Level 1 Sales Representative   
96               119      Gracy    Gellens               General Manager   
97               120      Giele   Laermans  Level 1 Sales Representative   
98               121   François    De Crée  Level 1 Sales Representative   

           WORK_PHONE  EXTENSION                FAX                   EMAIL  \
0   +33 

### Kolommen

Faxen zijn inmiddels ouderwets: niemand gebruikt zijn/haar faxnummer nog.<br>
Verwijder de bijbehorende kolom uit het DataFrame.

In [290]:
data_frame.drop(columns=['FAX'], inplace=True)
print(data_frame)

     SALES_STAFF_CODE FIRST_NAME  LAST_NAME                   POSITION_EN  \
0                   4      Denis       Pagé                Branch Manager   
1                   5  Élizabeth     Michel  Level 3 Sales Representative   
2                   6      Émile   Clermont  Level 1 Sales Representative   
3                   7    Étienne     Jauvin  Level 2 Sales Representative   
4                  12    Elsbeth  Wiesinger  Level 2 Sales Representative   
..                ...        ...        ...                           ...   
97                120      Giele   Laermans  Level 1 Sales Representative   
98                121   François    De Crée  Level 1 Sales Representative   
99                122     Yvette    Lattrez  Level 3 Sales Representative   
100               123      Willi  Seefelder  Level 2 Sales Representative   
101               124     Sabine     Grüner  Level 3 Sales Representative   

            WORK_PHONE  EXTENSION                   EMAIL  DATE_HIRED  \
0 